In [1]:
!git clone https://github.com/HIMASREE-2102/Image-Caption-Generator-CNN-LSTM.git

Cloning into 'Image-Caption-Generator-CNN-LSTM'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 14 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 218.60 KiB | 24.29 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [2]:
%cd /content/Image-Caption-Generator-CNN-LSTM

/content/Image-Caption-Generator-CNN-LSTM


In [3]:
!ls -lah

total 512K
drwxr-xr-x 3 root root 4.0K Oct 31 12:40 .
drwxr-xr-x 1 root root 4.0K Oct 31 12:40 ..
drwxr-xr-x 8 root root 4.0K Oct 31 12:40 .git
-rw-r--r-- 1 root root 4.6K Oct 31 12:40 .gitignore
-rw-r--r-- 1 root root 477K Oct 31 12:40 imagecaptiongenerator.ipynb
-rw-r--r-- 1 root root 1.1K Oct 31 12:40 LICENSE
-rw-r--r-- 1 root root 4.2K Oct 31 12:40 README.md


In [4]:
from google.colab import files

files.download('/content/Image-Caption-Generator-CNN-LSTM/imagecaptiongenerator.ipynb')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import json

path = "/content/Image-Caption-Generator-CNN-LSTM/imagecaptiongenerator.ipynb"

with open(path, "r") as f:
    nb = json.load(f)

print("Total cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"][:15]):
    print(f"\n{'='*50}")
    print("CELL:", i, "| TYPE:", cell["cell_type"])
    print("".join(cell["source"])[:1200])

Total cells: 52

CELL: 0 | TYPE: markdown
<a href="https://colab.research.google.com/github/HIMASREE-2102/Image-Caption-Generator-CNN-LSTM/blob/main/imagecaptiongenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CELL: 1 | TYPE: code
# Check GPU
import tensorflow as tf
print("TF version:", tf.__version__)
tf.config.list_physical_devices('GPU')


CELL: 2 | TYPE: code
!pip install -q kaggle


CELL: 3 | TYPE: code
from google.colab import files
files.upload()


CELL: 4 | TYPE: code
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


CELL: 5 | TYPE: code
!kaggle datasets download -d adityajn105/flickr8k


CELL: 6 | TYPE: code
!unzip -q flickr8k.zip


CELL: 7 | TYPE: code
!ls


CELL: 8 | TYPE: code
import string
import re

captions_file = 'captions.txt'  # change if needed

# Load raw captions
def load_doc(filename):
    with open(filename, 'r') as f:
        text = f.read()
    return

In [7]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"mirzashanehyder","key":"26998da7564fa43df10d26a5ea366215"}'}

In [8]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [9]:
!kaggle datasets list -s flickr8k

ref                                     title                                       size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------  ------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
shadabhussain/flickr8k                  Flickr8K                              2285754264  2019-03-24 17:53:27.480000          14007        149  0.1875           
sayanf/flickr8k                         Flickr8K                              1114642008  2021-02-16 17:23:04.523000           3781         14  0.6875           
aladdinpersson/flickr8kimagescaptions   Flickr8k-Images-Captions              1113116819  2020-07-12 09:20:01.120000           8125         84  0.5625           
adityajn105/flickr8k                    Flickr 8k Dataset                     1112971163  2020-04-27 07:27:19.103000         178867        602  0.75             
dibyansudiptiman/flickr-8k  

In [10]:
!kaggle datasets download -d adityajn105/flickr8k

Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k
License(s): CC0-1.0
100% 1.04G/1.04G [00:59<00:00, 18.6MB/s]



In [11]:
!unzip -q flickr8k.zip

In [12]:
import os

print("Images directory exists:", os.path.exists("Images"))
print("Captions file exists:", os.path.exists("captions.txt"))

if os.path.exists("Images"):
    print("Number of images:", len(os.listdir("Images")))

Images directory exists: True
Captions file exists: True
Number of images: 8091


In [13]:
for i, cell in enumerate(nb["cells"][15:], start=15):
    print(f"\n{'='*50}")
    print(f"CELL {i} | {cell['cell_type']}")
    print("".join(cell["source"])[:2000])


CELL 15 | code
def extract_features(directory):
    features = {}
    for filename in os.listdir(directory):
        if not filename.lower().endswith(('jpg', 'jpeg', 'png')):
            continue

        img_path = os.path.join(directory, filename)
        image = load_img(img_path, target_size=(299, 299))
        image = img_to_array(image)
        image = np.expand_dims(image, axis=0)
        image = preprocess_input(image)

        feature = cnn_model.predict(image, verbose=0)
        features[filename.split('.')[0]] = feature[0]

    return features

features = extract_features(IMAGE_DIR)
print("Total extracted features:", len(features))


CELL 16 | code
import pickle
with open('features.pkl', 'wb') as f:
    pickle.dump(features, f)


CELL 17 | code
from tensorflow.keras.preprocessing.text import Tokenizer

all_captions = []
for key in train_descriptions:
    all_captions.extend(train_descriptions[key])

tokenizer = Tokenizer(oov_token="<unk>")
tokenizer.fit_on_texts(all_caption

In [14]:
!pip -q install transformers sentencepiece accelerate gradio

In [15]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

caption_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

print("BLIP model loaded successfully!")

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

BLIP model loaded successfully!


In [17]:
import gradio as gr

def generate_caption(image):
    if image is None:
        return "Please upload an image"

    inputs = processor(
        image.convert("RGB"),
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = caption_model.generate(
            **inputs,
            num_beams=5,
            max_new_tokens=30
        )

    return processor.decode(
        output[0],
        skip_special_tokens=True
    )

ui = gr.Interface(
    fn=generate_caption,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Textbox(label="Generated Caption"),
    title="AI Image Caption Generator",
    description="Upload an image to generate an AI caption."
)

ui.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f32bea453edc73d9f4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
